# ML-Driven DCF Valuation Engine -- Part 1: Data & Feature Engineering

**An institutional-grade pipeline that uses Machine Learning to forecast business drivers and compute the intrinsic value of publicly traded companies.**

```
FMP Financial Data          <-- YOU ARE HERE
        |
Merge Financial Statements  <-- YOU ARE HERE
        |
Feature Engineering         <-- YOU ARE HERE
        |
Train ML Models             (Part 2)
        |
Forecast & DCF Valuation    (Part 2)
        |
Compare vs Market Price     (Part 3)
        |
Buy / Hold / Sell Signal    (Part 3)
```

---
**Data Source:** Financial Modeling Prep (FMP) API -- 14 Large-Cap US Equities, 5 Years of Annual Filings  
**Model:** Random Forest Regressor (one per business driver)  
**Valuation:** 5-Year Explicit Forecast + Gordon Growth Terminal Value  

---
## Step 1 -- FMP Financial Data
Load the individual financial statement CSVs that were pulled from the FMP API.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# -- Load each financial statement from its own CSV --
income     = pd.read_csv('fmp_dataset/csv/income-statement.csv')
balance    = pd.read_csv('fmp_dataset/csv/balance-sheet-statement.csv')
cashflow   = pd.read_csv('fmp_dataset/csv/cash-flow-statement.csv')
enterprise = pd.read_csv('fmp_dataset/csv/enterprise-values.csv')
quote      = pd.read_csv('fmp_dataset/csv/quote.csv')
dcf_fmp    = pd.read_csv('fmp_dataset/csv/discounted-cash-flow.csv')

print(f'Income Statement   : {income.shape}')
print(f'Balance Sheet      : {balance.shape}')
print(f'Cash Flow Statement: {cashflow.shape}')
print(f'Enterprise Values  : {enterprise.shape}')
print(f'Live Quotes        : {quote.shape}')
print(f'FMP DCF (benchmark): {dcf_fmp.shape}')
print(f'\nTickers: {sorted(income["symbol"].unique())}')

Income Statement   : (70, 40)
Balance Sheet      : (70, 62)
Cash Flow Statement: (70, 48)
Enterprise Values  : (70, 9)
Live Quotes        : (14, 18)
FMP DCF (benchmark): (14, 5)

Tickers: ['AAPL', 'AMD', 'AMZN', 'COST', 'GOOGL', 'INTC', 'JPM', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA', 'V', 'WMT']


---
## Step 2 -- Merge Financial Statements
Join Income Statement + Balance Sheet + Cash Flow + Enterprise Values on `(symbol, date)` to create one row per company-year with all the data we need.

In [2]:
# -- Select only the columns we actually need from each statement --
# This keeps the merge clean and avoids duplicate/ambiguous columns.

inc_cols = [
    'date', 'symbol', 'revenue', 'costOfRevenue', 'grossProfit',
    'researchAndDevelopmentExpenses', 'sellingGeneralAndAdministrativeExpenses',
    'operatingExpenses', 'operatingIncome', 'ebit', 'ebitda',
    'depreciationAndAmortization', 'interestExpense',
    'incomeBeforeTax', 'incomeTaxExpense', 'netIncome',
    'eps', 'epsDiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil'
]

bs_cols = [
    'date', 'symbol', 'cashAndCashEquivalents', 'shortTermInvestments',
    'netReceivables', 'inventory', 'totalCurrentAssets',
    'propertyPlantEquipmentNet', 'goodwillAndIntangibleAssets',
    'totalAssets', 'accountPayables', 'totalCurrentLiabilities',
    'longTermDebt', 'totalLiabilities', 'totalStockholdersEquity',
    'totalDebt', 'netDebt'
]

cf_cols = [
    'date', 'symbol', 'stockBasedCompensation', 'changeInWorkingCapital',
    'operatingCashFlow', 'capitalExpenditure', 'freeCashFlow',
    'acquisitionsNet', 'netCashProvidedByOperatingActivities',
    'commonStockRepurchased', 'commonDividendsPaid'
]

ev_cols = [
    'date', 'symbol', 'stockPrice', 'numberOfShares',
    'marketCapitalization', 'enterpriseValue'
]

# -- Merge on (symbol, date) using inner joins --
merged = income[inc_cols].merge(balance[bs_cols],  on=['symbol', 'date'], how='inner')
merged = merged.merge(cashflow[cf_cols],           on=['symbol', 'date'], how='inner')
merged = merged.merge(enterprise[ev_cols],         on=['symbol', 'date'], how='inner')

# -- Sort chronologically within each company --
merged['date'] = pd.to_datetime(merged['date'])
merged = merged.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f'Merged dataset: {merged.shape[0]} rows x {merged.shape[1]} columns')
print(f'Companies: {merged["symbol"].nunique()}  |  Years per company: ~{merged.shape[0] // merged["symbol"].nunique()}')
merged.head(3)

Merged dataset: 70 rows x 48 columns
Companies: 14  |  Years per company: ~5


,date,symbol,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,sellingGeneralAndAdministrativeExpenses,operatingExpenses,operatingIncome,ebit,...,capitalExpenditure,freeCashFlow,acquisitionsNet,netCashProvidedByOperatingActivities,commonStockRepurchased,commonDividendsPaid,stockPrice,numberOfShares,marketCapitalization,enterpriseValue
0,2021-09-25,AAPL,365817000000,212981000000,152836000000,21914000000,21973000000,43887000000,108949000000,111852000000,...,-11085000000,92953000000,-33000000,104038000000,-85971000000,-14467000000,146.92,16701272000,2453750882240,2555332882240
1,2022-09-24,AAPL,394328000000,223546000000,170782000000,26251000000,25094000000,51345000000,119437000000,122034000000,...,-10708000000,111443000000,-306000000,122151000000,-89402000000,-14841000000,150.43,16215963000,2439367314090,2548201314090
2,2023-09-30,AAPL,383285000000,214137000000,169148000000,29915000000,24932000000,54847000000,114301000000,117669000000,...,-10959000000,99584000000,0,110543000000,-77550000000,-15025000000,171.21,15744231000,2695569789510,2789534789510


---
## Step 3 -- Feature Engineering
Calculate the **business drivers** that the DCF model needs. These are the ratios and growth rates that, when forecasted, let us reconstruct a full set of projected financial statements.

| Driver | Formula | Why It Matters |
|--------|---------|----------------|
| Revenue Growth | `(Rev_t / Rev_t-1) - 1` | Top-line trajectory |
| Gross Margin | `Gross Profit / Revenue` | Pricing power & COGS efficiency |
| EBIT Margin | `EBIT / Revenue` | Core operating profitability |
| Effective Tax Rate | `Tax / Pre-Tax Income` | Cash tax leakage |
| D&A % Revenue | `D&A / Revenue` | Asset intensity |
| Capex % Revenue | `|Capex| / Revenue` | Reinvestment rate |
| NWC % Revenue | `(Current Assets - Current Liabilities) / Revenue` | Working capital needs |

In [3]:
def engineer_features(group):
    """Calculate business drivers for a single company's time-series."""
    g = group.copy()
    rev = g['revenue']

    # -- Growth --
    g['rev_growth'] = rev.pct_change()

    # -- Margins --
    g['gross_margin']  = g['grossProfit'] / rev
    g['ebit_margin']   = g['ebit'] / rev

    # -- Tax Rate (clipped to 0-40% to remove noise from loss years) --
    g['eff_tax_rate'] = (g['incomeTaxExpense'] / g['incomeBeforeTax']).clip(0, 0.40)

    # -- Capital Intensity --
    g['da_pct_rev']    = g['depreciationAndAmortization'] / rev
    g['capex_pct_rev'] = g['capitalExpenditure'].abs() / rev

    # -- Working Capital --
    g['nwc']           = g['totalCurrentAssets'] - g['totalCurrentLiabilities']
    g['nwc_pct_rev']   = g['nwc'] / rev
    g['delta_nwc']     = g['nwc'].diff()

    # -- Lagged features (previous year's drivers -> this year's input) --
    driver_cols = ['rev_growth', 'gross_margin', 'ebit_margin',
                   'eff_tax_rate', 'da_pct_rev', 'capex_pct_rev', 'nwc_pct_rev']
    for col in driver_cols:
        g[f'{col}_lag1'] = g[col].shift(1)

    return g

merged = merged.groupby('symbol', group_keys=False).apply(engineer_features)

# -- Clean up infinities and NaNs from pct_change / division --
merged = merged.replace([np.inf, -np.inf], np.nan)

print('Feature engineering complete.')
print(f'Columns added: rev_growth, gross_margin, ebit_margin, eff_tax_rate, da_pct_rev, capex_pct_rev, nwc_pct_rev + lags')

# Show a snapshot of the drivers for one company
driver_preview = ['date', 'symbol', 'revenue', 'rev_growth', 'ebit_margin',
                  'eff_tax_rate', 'da_pct_rev', 'capex_pct_rev', 'nwc_pct_rev']
merged[merged['symbol'] == 'AAPL'][driver_preview]

Feature engineering complete.
Columns added: rev_growth, gross_margin, ebit_margin, eff_tax_rate, da_pct_rev, capex_pct_rev, nwc_pct_rev + lags


,date,symbol,revenue,rev_growth,ebit_margin,eff_tax_rate,da_pct_rev,capex_pct_rev,nwc_pct_rev
0,2021-09-25,AAPL,365817000000,NaN,0.305759,0.133023,0.030846,0.030302,0.025573
1,2022-09-24,AAPL,394328000000,0.077938,0.309473,0.162045,0.028159,0.027155,-0.047111
2,2023-09-30,AAPL,383285000000,-0.028005,0.307001,0.147192,0.030053,0.028592,-0.004545
3,2024-09-28,AAPL,391035000000,0.020220,0.315790,0.240912,0.029268,0.024159,-0.059854
4,2025-09-27,AAPL,416161000000,0.064255,0.318937,0.156100,0.028109,0.030553,-0.042469
